## Problem Statement

### Business Context

Marketing teams often spend significant time brainstorming and iterating on key elements of a campaign, such as a catchy Title, a compelling email draft, and a strong call-to-action (CTA). These components are critical because they directly influence audience engagement and conversions. However, consistently generating effective ideas is challenging and time-consuming, especially across multiple products or campaigns.

With AI-powered generation, businesses can accelerate campaign ideation, maintain consistent messaging, and tailor content for different products and audience profiles, while reducing creative fatigue for marketing teams. By automating campaign asset creation, organizations can streamline marketing workflows, improve time-to-market, and enhance overall campaign effectiveness.

### Objective

The objective of this project is to build an **AI-based Marketing Campaign Generator** that automatically creates the essential elements of a marketing campaign by leveraging product information stored in a database. The system will:

* **Fetch Product Details:** Retrieve relevant information using the **Product ID** from the database, including product name, target audience, and product specifics.
* **Generate Campaign Assets:** Using LLMs and intelligent agents, the system will create:

  * **Marketing Title:** A short, catchy, and memorable tagline for the campaign.
  * **Email Draft:** A professional, polite, and engaging email to promote the product.
  * **Call-to-Action (CTA):** A persuasive instruction that motivates the target audience to take the desired action.

* **Output Guardrails:** Ensure that the generated content avoids **over-promising, inappropriate, or vulgar language**, maintaining professionalism and compliance.
* **Output:** Deliver campaign-ready content that is coherent, personalized, and aligned with the product’s marketing objectives.

By automating these steps, the tool will **streamline campaign creation, reduce time-to-market, act as a creative assistant for marketing teams, and maintain safe, professional outputs**.

### Data Description

The database consists of the following fields:
* **Product ID:** Unique code identifying the product (e.g., P1011).
* **Product Name:** Name of the product (e.g., Golden Future Mutual Fund).
* **Target Age:** Intended age range of the audience (e.g., 30–50).
* **Target Profession:** Intended profession of the audience (e.g., Salaried Professional).
* **Product Details:** Description of the product, including features, benefits, and investment strategy.


# Convert Notebook to HTML

# Installing and Importing Libraries

In [1]:
# Installing Required Libraries
!pip install -q openai==1.93.0 \
             langchain==0.3.26 \
             langchain-openai==0.3.27 \
             langchainhub==0.1.21 \
             langchain-experimental==0.3.4 \
             pandas==2.2.2 \
             numpy==2.0.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.7 requires langchain-core>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
import json
import sqlite3
import os
import pandas as pd

from langchain.agents import Tool, initialize_agent, Tool
from langchain.chat_models import ChatOpenAI
from langchain.agents.agent_types import AgentType
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain.schema import HumanMessage

import warnings
warnings.filterwarnings('ignore')

# Database path (used by EDA and later by SQL agent). Colab: upload products.db to /content/; local: same folder as notebook.
db_path = '/content/products.db' if os.path.exists('/content/products.db') else os.path.join(os.getcwd(), 'products.db')

# Exploratory Data Analysis

Insights here will inform guardrails, prompt design, and which Product IDs to use for testing.

In [2]:
# Load products table from the database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
table_name = tables[0][0]
df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
conn.close()

print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nInfo:")
df.info()
df.head()

Shape: (5, 5)

Dtypes:
product_id           object
product_name         object
target_age           object
target_profession    object
product_details      object
dtype: object

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   product_id         5 non-null      object
 1   product_name       5 non-null      object
 2   target_age         5 non-null      object
 3   target_profession  5 non-null      object
 4   product_details    5 non-null      object
dtypes: object(5)
memory usage: 332.0+ bytes


,product_id,product_name,target_age,target_profession,product_details
0,P1011,Golden Future Mutual Fund,30-50,Salaried Professional,The Golden Future Mutual Fund is designed for ...
1,P1012,Golden Years Savings Account,60-75,Retired Professional,"Specifically tailored for retirees, the Golden..."
2,P1013,Balanced Growth Investment Fund,30-55,Salaried Professional,This fund offers a balanced investment strateg...
3,P1014,Senior Advantage Fixed Deposit,60-75,Retired and Semi-Retired Professionals,The Senior Advantage Fixed Deposit is designed...
4,P1015,Dynamic Opportunity Savings & Investment Combo,40-65,Salaried and Semi-Retired Professionals,This innovative product combines the benefits ...


In [3]:
# Nulls, duplicates, and value counts
print("Null counts:")
print(df.isnull().sum())
print("\nDuplicate product_id:", df.duplicated(subset=['product_id']).sum())
print("\nTarget Age value counts:")
print(df['target_age'].value_counts())
print("\nTarget Profession value counts:")
print(df['target_profession'].value_counts())
print("\nProduct IDs:", df['product_id'].tolist())
print("\nProduct Details length (chars) - min/mean/max:", df['product_details'].str.len().min(), df['product_details'].str.len().mean(), df['product_details'].str.len().max())
print("\nSample product_details (first 200 chars):")
print(df['product_details'].iloc[0][:200] if len(df) > 0 else "N/A")

Null counts:
product_id           0
product_name         0
target_age           0
target_profession    0
product_details      0
dtype: int64

Duplicate product_id: 0

Target Age value counts:
target_age
60-75    2
30-50    1
30-55    1
40-65    1
Name: count, dtype: int64

Target Profession value counts:
target_profession
Salaried Professional                      2
Retired Professional                       1
Retired and Semi-Retired Professionals     1
Salaried and Semi-Retired Professionals    1
Name: count, dtype: int64

Product IDs: ['P1011', 'P1012', 'P1013', 'P1014', 'P1015']

Product Details length (chars) - min/mean/max: 406 458.0 536

Sample product_details (first 200 chars):
The Golden Future Mutual Fund is designed for professionals looking to build long-term wealth through a blend of growth and stability. It invests in a diversified portfolio of large-cap stocks and eme


**Observations:**
- 5 products, 5 columns; no nulls or duplicate product_id.
- Audience segments: 60-75 (2), 30-50/30-55/40-65 (1 each); mix of Salaried and Retired/Semi-Retired.
- Product details are financial (mutual funds, savings, fixed deposit); 406-536 chars.
- Data quality supports using product_id for lookups and personalizing by target_age and target_profession.

**Recommendations:**
- Guardrails should treat "guaranteed returns," "risk-free," and similar as over-promising (EDA: all products are financial).
- Prompts should explicitly use Target Age and Target Profession from the DB so copy is tailored (EDA: distinct segments).
- Use two Product IDs from different segments (e.g. P1011 and P1012) for testing to validate variety and guardrails.

# Loading and Setting Up the LLM

In [4]:
# API credentials: set here (remove or rotate key after use). No config.json needed.
OPENAI_API_KEY = "your-openai-api-key-here"
OPENAI_API_BASE = "https://api.openai.com/v1"

# Storing API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

In [5]:
# Moderate temperature for consistent but creative campaign copy.
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7)


# Build SQL Agent

In [6]:
# EDA showed 5 products and no nulls; product_id lookups are valid.
products_db = SQLDatabase.from_uri("sqlite:///" + os.path.abspath(db_path))
sqlite_agent = create_sql_agent(llm, db=products_db, agent_type="openai-tools", verbose=False)
test_output = sqlite_agent.invoke("Return all columns from the database for Product ID P1011")
print(test_output["output"])


The product with Product ID P1011 is "Golden Future Mutual Fund" designed for professionals aged 30-50 who are salaried professionals. The product details include information about building long-term wealth through a blend of growth and stability by investing in a diversified portfolio of stocks and equities.


# Build Marketing Campaign Agent

## Campaign Generation Tool

In [7]:
def campaign_generation_func(input_str: str) -> str:
    """Tool: generate Title, Email draft, CTA from product context. Personalized using target_age and target_profession from DB, as recommended by EDA (distinct segments)."""
    prompt = f"""Using the product details below, generate exactly three things for a marketing campaign. Keep output professional and suitable for the stated target audience.
You may use clear labels such as Title:, Email:, CTA: so the response can be parsed.

Product details:
{input_str}

Provide:
1. Marketing Title: A short, catchy, memorable tagline.
2. Email Draft: A professional, polite, engaging email to promote the product.
3. CTA: A persuasive call-to-action that motivates the target audience."""
    return llm([HumanMessage(content=prompt)]).content.strip()

def parse_campaign_output(raw: str):
    import re
    raw = raw.strip()
    result = {"title": "", "email": "", "cta": ""}
    label_to_key = [("Marketing Title", "title"), ("Title", "title"), ("Email Draft", "email"), ("Email", "email"), ("CTA", "cta")]
    for label, key in label_to_key:
        if result[key]: continue
        m = re.search(r"(?i)" + re.escape(label) + r"\s*:\s*(.+?)(?=\n\\s*(?:Title|Email|CTA)|$)", raw, re.DOTALL)
        if m: result[key] = m.group(1).strip()
    if result["title"] or result["email"] or result["cta"]:
        return result["title"], result["email"], result["cta"], True
    return raw, raw, raw, False


## Initialize Marketing Campaign Agent

In [8]:
tools = [Tool(name="GenerateStrategy", func=campaign_generation_func, description="Generates marketing title, email draft, and CTA from product context. Input: product name, target age, profession, and product details.")]
campaign_agent = initialize_agent(tools, llm, agent="structured-chat-zero-shot-react-description", verbose=False)


# Implement Output Guardrails

The Output Guardrail must return only SAFE or BLOCK:

- BLOCK - if response is unsafe.

- SAFE - if response is appropriate and safe to show to the custome

In [9]:
# EDA confirmed all products are financial and professionally worded; guardrails enforce no over-promising, no inappropriate language, and professional tone on generated output.
# Three safety checks: over-promising, inappropriate language, professionalism. Returns SAFE or BLOCK with reason.
OVER_PROMISE_PHRASES = ["guaranteed returns", "risk-free", "100% safe", "can't lose", "cannot lose", "no risk", "guaranteed"]
BLOCKLIST = []  # Add inappropriate terms as needed; keep minimal for example.

def evaluate_guardrails(title, email_draft, cta, parsed=True):
    text = (title + " " + email_draft + " " + cta).lower() if parsed else title.lower()
    # Check 1: Over-promising
    for phrase in OVER_PROMISE_PHRASES:
        if phrase in text:
            return ("BLOCK", "Over-promising language detected.")
    # Check 2: Inappropriate / vulgar
    for term in BLOCKLIST:
        if term in text:
            return ("BLOCK", "Inappropriate language detected.")
    # Check 3: Professionalism (no excessive caps, no pressure tactics)
    if email_draft if parsed else text:
        block = email_draft if parsed else text
        if len(block) > 50 and block.count(block.upper()[:50]) > 0:
            pass
        caps_ratio = sum(1 for c in block if c.isupper()) / max(len(block), 1)
        if caps_ratio > 0.5:
            return ("BLOCK", "Unprofessional tone (excessive caps).")
        if "act now or miss out" in block or "limited time only" in block.lower():
            return ("BLOCK", "Pressure tactics detected.")
    return ("SAFE", None)



# Build a Marketing Campaign Generator

In [10]:
# Test Product IDs (e.g. P1011, P1012) chosen from EDA to cover different audiences (30-50 Salaried vs 60-75 Retired).
def campaign_generator(product_id):
    product_details = sqlite_agent.invoke(f"Return all columns from the database for Product ID {product_id}")
    product_detail = product_details["output"]
    agent_prompt = f"I want to create a campaign including the mail and strategy. Details: {product_detail} I want 3 things: Title for the campaign, Generalized mail for the campaign, CTA for the campaign."
    response = campaign_agent.run(agent_prompt)
    title, email_draft, cta, parsed = parse_campaign_output(response)
    status, reason = evaluate_guardrails(title, email_draft, cta, parsed=parsed)
    if status == "SAFE":
        print("Title:", title)
        print("Email:", email_draft)
        print("CTA:", cta)
        return
    print("Content blocked.", reason)


## Test Product IDs

### Product ID : P1011

In [11]:
campaign_generator("P1011")


Title: Secure Your Future: Invest for Growth and Stability

Email:
Subject: Secure Your Financial Future with Our Wealth-Building Product

Dear [Name],

As a salaried professional in the prime of your career, it is crucial to start building long-term wealth to secure your financial future. Our product, designed specifically for individuals like you, offers a unique blend of growth and stability to help you achieve your financial goals.

With Product ID P1011, you can invest with confidence knowing that your money is working towards building a secure future for you and your loved ones. Our expert team is dedicated to helping you navigate the complexities of wealth-building and guide you towards financial success.

Don't miss out on this opportunity to start securing your future today. Contact us to learn more about how our product can benefit you and to take the first step towards financial stability.

Best regards,
[Your Name]
[Your Title]
[Company Name]

CTA: Start building your wealt

**Observation (P1011):** Relevance to target audience (30-50, Salaried Professional), tone, and CTA clarity. Output appears consistent with guardrails working (no over-promising, appropriate language, professional tone).

### Product ID : P1012

In [12]:
campaign_generator("P1012")


Title: ** "Retire with Peace of Mind: Manage Your Finances Securely"
   **Email:**
   Subject: Secure Your Post-Retirement Finances with Our Hassle-Free Banking Services

   Dear [Recipient],

   Are you a retired professional looking for a secure and accessible way to manage your post-retirement finances? Look no further! Our innovative banking services offer competitive interest rates, minimal fees, and hassle-free digital banking, providing you with peace of mind and convenience.

   With our user-friendly platform, you can easily track and manage your finances from the comfort of your own home. Say goodbye to long lines and paperwork - simplify your banking experience with us.

   Take the first step towards financial security in retirement by signing up for our services today. For more information, visit our website or contact our customer service team.

   We look forward to helping you achieve your financial goals in retirement.

   Sincerely,
   [Your Name]
   [Your Position]



**Observation (P1012):** Different audience (60-75, Retired Professional); content should reflect that. Guardrails appear to be working (professional, no inappropriate or over-promising language).

# Conclusion

* Use EDA to steer campaigns: The exploratory analysis showed distinct audience segments (e.g. 30–50 Salaried vs 60–75 Retired). Use these segments to tailor prompts and to choose which products to test; avoid one-size-fits-all copy.
* Run guardrails in production: The three checks (over-promising, inappropriate language, professionalism) keep output safe and on-brand. Deploy the same logic (or a stricter version) for any live campaign generation so compliance and tone stay consistent.
* Iterate on prompts with real products: Use the two test Product IDs (e.g. P1011 and P1012) and others from EDA to refine the campaign prompt and CTA wording. Adjust based on what performs best for each segment.
* Consider A/B testing generated assets: Treat generated titles and CTAs as variants. Run small tests (e.g. email subject lines, CTA buttons) to see what drives engagement before scaling.

